### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
> 4. Submit your homework as html file just like our exercise did.
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [1]:
from collections import Counter
import json
from pathlib import Path

DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "enwiki-train.json"
TEST_PATH = DATA_DIR / "enwiki-test.json"


def load_jsonl(file_path: Path):
    records = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def count_documents_by_label(records):
    return Counter(record["label"] for record in records)


train_records = load_jsonl(TRAIN_PATH)
test_records = load_jsonl(TEST_PATH)
train_label_counts = count_documents_by_label(train_records)
test_label_counts = count_documents_by_label(test_records)

print("Training dataset - number of documents in each class:")
for label, count in sorted(train_label_counts.items()):
    print(f"{label}: {count}")

print("\nTesting dataset - number of documents in each class:")
for label, count in sorted(test_label_counts.items()):
    print(f"{label}: {count}")


Training dataset - number of documents in each class:
Actor: 79
Animal: 82
Artist: 457
Book: 858
Disease: 202
Film: 2752
Food: 121
Politician: 3441
Software: 239
Writer: 769

Testing dataset - number of documents in each class:
Actor: 1
Animal: 11
Artist: 63
Book: 117
Disease: 18
Film: 296
Food: 16
Politician: 383
Software: 27
Writer: 68


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [2]:
import spacy
from collections import defaultdict

nlp = spacy.blank("en")
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")


def average_sentence_count_by_label(records, nlp, batch_size: int = 64):
    sentence_totals = defaultdict(int)
    document_totals = defaultdict(int)

    for record, doc in zip(records, nlp.pipe((record["text"] for record in records), batch_size=batch_size)):
        label = record["label"]
        sentence_totals[label] += sum(1 for sent in doc.sents if sent.text.strip())
        document_totals[label] += 1

    return {
        label: sentence_totals[label] / document_totals[label]
        for label in sorted(document_totals)
    }


train_avg_sentence_counts = average_sentence_count_by_label(train_records, nlp)
test_avg_sentence_counts = average_sentence_count_by_label(test_records, nlp)

print("Training dataset - average number of sentences in each class:")
for label, avg_count in train_avg_sentence_counts.items():
    print(f"{label}: {avg_count:.2f}")

print("\nTesting dataset - average number of sentences in each class:")
for label, avg_count in test_avg_sentence_counts.items():
    print(f"{label}: {avg_count:.2f}")


Training dataset - average number of sentences in each class:
Actor: 69.33
Animal: 67.06
Artist: 185.94
Book: 205.08
Disease: 347.86
Film: 177.88
Food: 153.44
Politician: 222.86
Software: 201.13
Writer: 216.11

Testing dataset - average number of sentences in each class:
Actor: 177.00
Animal: 62.82
Artist: 174.97
Book: 197.96
Disease: 325.89
Film: 173.68
Food: 165.12
Politician: 233.03
Software: 204.00
Writer: 220.60


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [ ]:
def average_token_count_by_label(records, nlp, batch_size: int = 64):
    token_totals = defaultdict(int)
    document_totals = defaultdict(int)

    for record, doc in zip(records, nlp.pipe((record["text"] for record in records), batch_size=batch_size)):
        label = record["label"]
        token_totals[label] += sum(1 for token in doc if not token.is_space)
        document_totals[label] += 1

    return {
        label: token_totals[label] / document_totals[label]
        for label in sorted(document_totals)
    }


train_avg_token_counts = average_token_count_by_label(train_records, nlp)
test_avg_token_counts = average_token_count_by_label(test_records, nlp)

print("Training dataset - average number of tokens in each class:")
for label, avg_count in train_avg_token_counts.items():
    print(f"{label}: {avg_count:.2f}")

print("\nTesting dataset - average number of tokens in each class:")
for label, avg_count in test_avg_token_counts.items():
    print(f"{label}: {avg_count:.2f}")


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [ ]:
import re

TOKEN_PATTERN = re.compile(r"[a-z0-9]+")


def preprocess_sentence(sentence_text: str):
    return TOKEN_PATTERN.findall(sentence_text.lower())


def preprocess_document(text: str, nlp):
    processed_sentences = []
    doc = nlp(text)

    for sent in doc.sents:
        cleaned_tokens = preprocess_sentence(sent.text)
        if cleaned_tokens:
            processed_sentences.append(cleaned_tokens)

    return processed_sentences


def attach_processed_text(records, nlp):
    for record in records:
        processed_sentences = preprocess_document(record["text"], nlp)
        processed_tokens = [token for sentence in processed_sentences for token in sentence]

        record["processed_sentences"] = processed_sentences
        record["processed_tokens"] = processed_tokens
        record["processed_text"] = " ".join(processed_tokens)


def get_first_record_of_each_class(records):
    first_records = {}
    for record in records:
        label = record["label"]
        if label not in first_records:
            first_records[label] = record
    return first_records


attach_processed_text(train_records, nlp)
attach_processed_text(test_records, nlp)

train_first_records = get_first_record_of_each_class(train_records)
test_first_records = get_first_record_of_each_class(test_records)

print("Training dataset - first article in each class and its first 40 processed words:")
for label in sorted(train_first_records):
    record = train_first_records[label]
    preview_words = " ".join(record["processed_tokens"][:40])
    print(f"{label} | title: {record['title']}")
    print(f"first 40 words: {preview_words}")
    print()

print("Testing dataset - first article in each class and its first 40 processed words:")
for label in sorted(test_first_records):
    record = test_first_records[label]
    preview_words = " ".join(record["processed_tokens"][:40])
    print(f"{label} | title: {record['title']}")
    print(f"first 40 words: {preview_words}")
    print()


### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [ ]:
import math
import random

if "processed_sentences" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_sentences" not in test_records[0]:
    attach_processed_text(test_records, nlp)


def collect_all_sentences(records):
    all_sentences = []
    for record in records:
        all_sentences.extend(record["processed_sentences"])
    return all_sentences


train_sentences_raw = collect_all_sentences(train_records)
test_sentences_raw = collect_all_sentences(test_records)

UNK_TOKEN = "<UNK>"
START_TOKEN = "<s>"
END_TOKEN = "</s>"
MIN_FREQUENCY = 10
ADDITIVE_ALPHA = 1.0


def build_vocabulary(sentences, min_frequency: int = 10):
    token_counter = Counter()
    for sentence in sentences:
        token_counter.update(sentence)

    vocabulary = {
        token
        for token, count in token_counter.items()
        if count >= min_frequency
    }
    vocabulary.update({UNK_TOKEN, START_TOKEN, END_TOKEN})
    return vocabulary, token_counter


def replace_rare_and_oov_tokens(sentences, vocabulary):
    normalized_sentences = []
    for sentence in sentences:
        normalized_sentences.append([
            token if token in vocabulary else UNK_TOKEN
            for token in sentence
        ])
    return normalized_sentences


vocabulary, train_token_counter = build_vocabulary(train_sentences_raw, min_frequency=MIN_FREQUENCY)
train_sentences = replace_rare_and_oov_tokens(train_sentences_raw, vocabulary)
test_sentences = replace_rare_and_oov_tokens(test_sentences_raw, vocabulary)
vocabulary_list = sorted(vocabulary)
vocabulary_size = len(vocabulary_list)


class AdditiveNGramLanguageModel:
    def __init__(self, n: int, vocabulary, alpha: float = 1.0):
        self.n = n
        self.alpha = alpha
        self.vocabulary = set(vocabulary)
        self.vocabulary_list = sorted(vocabulary)
        self.vocabulary_size = len(self.vocabulary_list)
        self.ngram_counts = Counter()
        self.context_counts = Counter()

    def pad_sentence(self, sentence):
        if self.n == 1:
            return sentence + [END_TOKEN]
        start_tokens = [START_TOKEN] * (self.n - 1)
        return start_tokens + sentence + [END_TOKEN]

    def fit(self, sentences):
        for sentence in sentences:
            padded_sentence = self.pad_sentence(sentence)
            for i in range(len(padded_sentence) - self.n + 1):
                ngram = tuple(padded_sentence[i : i + self.n])
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

    def get_probability(self, token, context=None):
        if context is None:
            context = ()
        context = tuple(context[-(self.n - 1) :]) if self.n > 1 else ()
        if token not in self.vocabulary:
            token = UNK_TOKEN

        ngram = context + (token,)
        numerator = self.ngram_counts[ngram] + self.alpha
        denominator = self.context_counts[context] + self.alpha * self.vocabulary_size
        return numerator / denominator

    def sentence_log_probability(self, sentence):
        padded_sentence = self.pad_sentence(sentence)
        log_probability = 0.0
        for i in range(len(padded_sentence) - self.n + 1):
            ngram = tuple(padded_sentence[i : i + self.n])
            log_probability += math.log(self.get_probability(ngram[-1], ngram[:-1]))
        return log_probability

    def perplexity(self, sentences):
        total_log_probability = 0.0
        total_predicted_tokens = 0
        for sentence in sentences:
            padded_sentence = self.pad_sentence(sentence)
            total_log_probability += self.sentence_log_probability(sentence)
            total_predicted_tokens += len(padded_sentence) - self.n + 1
        return math.exp(-total_log_probability / total_predicted_tokens)

    def generate_sentence(self, max_length: int = 20, random_seed=None):
        if random_seed is not None:
            random.seed(random_seed)

        generated_tokens = []
        context = [START_TOKEN] * (self.n - 1) if self.n > 1 else []
        candidate_tokens = [token for token in self.vocabulary_list if token != START_TOKEN]

        for _ in range(max_length):
            probabilities = [self.get_probability(token, context) for token in candidate_tokens]
            next_token = random.choices(candidate_tokens, weights=probabilities, k=1)[0]
            if next_token == END_TOKEN:
                break
            generated_tokens.append(next_token)
            if self.n > 1:
                context = (context + [next_token])[-(self.n - 1) :]

        return generated_tokens


unigram_model = AdditiveNGramLanguageModel(n=1, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)
bigram_model = AdditiveNGramLanguageModel(n=2, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)
trigram_model = AdditiveNGramLanguageModel(n=3, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)

unigram_model.fit(train_sentences)
bigram_model.fit(train_sentences)
trigram_model.fit(train_sentences)

language_models = {
    "unigram": unigram_model,
    "bigram": bigram_model,
    "trigram": trigram_model,
}

print("Language model preprocessing and training are ready.")
print(f"Number of training sentences: {len(train_sentences)}")
print(f"Number of testing sentences: {len(test_sentences)}")
print(f"Raw training vocabulary size: {len(train_token_counter)}")
print(f"Final vocabulary size after cutoff={MIN_FREQUENCY}: {vocabulary_size}")
print(f"Additive smoothing alpha: {ADDITIVE_ALPHA}")


> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [ ]:
perplexity_results = {}

for model_name, model in language_models.items():
    perplexity_results[model_name] = model.perplexity(test_sentences)

print("Perplexity on the testing dataset:")
for model_name, perplexity_value in perplexity_results.items():
    print(f"{model_name}: {perplexity_value:.4f}")

sorted_results = sorted(perplexity_results.items(), key=lambda item: item[1])
best_model_name, best_perplexity = sorted_results[0]
worst_model_name, worst_perplexity = sorted_results[-1]

print("\nExplanation:")
print(f"The best model is {best_model_name} with perplexity {best_perplexity:.4f}.")
print(f"The weakest model is {worst_model_name} with perplexity {worst_perplexity:.4f}.")
print("Lower perplexity means the model predicts the testing sentences better.")
print("Unigram usually performs worst because it ignores context, while bigram and trigram use local context.")
print("If trigram is not better than bigram, the reason is often data sparsity.")


> 3) Use each built model to generate five sentences and explain these generated patterns.


In [ ]:
NUM_SENTENCES_TO_GENERATE = 5
MAX_GENERATED_LENGTH = 20


def tokens_to_sentence(tokens):
    if not tokens:
        return "[empty sentence]"
    return " ".join(tokens)


generated_sentences = {}

for model_index, (model_name, model) in enumerate(language_models.items(), start=1):
    model_sentences = []
    for sentence_index in range(NUM_SENTENCES_TO_GENERATE):
        random_seed = model_index * 100 + sentence_index + 1
        generated_tokens = model.generate_sentence(max_length=MAX_GENERATED_LENGTH, random_seed=random_seed)
        model_sentences.append(tokens_to_sentence(generated_tokens))
    generated_sentences[model_name] = model_sentences

print("Generated sentences from each language model:\n")
for model_name, sentence_list in generated_sentences.items():
    print(f"{model_name} model:")
    for sentence_number, sentence_text in enumerate(sentence_list, start=1):
        print(f"{sentence_number}. {sentence_text}")
    print()

print("Pattern analysis:")
print("1. Unigram usually produces the loosest sentences because it ignores word order.")
print("2. Bigram usually gives more reasonable local phrases because it uses one-word context.")
print("3. Trigram often looks more fluent because it uses two-word context.")
print("4. N-gram models may still repeat words or lose global coherence.")
print("5. Strange trigram output is often caused by data sparsity.")


> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [ ]:
import shutil
import subprocess
from pathlib import Path

try:
    import kenlm
except ImportError as exc:
    raise ImportError("kenlm Python package is not installed. Please install kenlm first before running this cell.") from exc


def ensure_command_exists(command_name: str):
    command_path = shutil.which(command_name)
    if command_path is None:
        raise FileNotFoundError(f"Command '{command_name}' was not found. Please install kenlm binaries and make sure this command is in PATH.")
    return command_path


lmplz_path = ensure_command_exists("lmplz")
build_binary_path = ensure_command_exists("build_binary")

KENLM_DIR = Path("kenlm_artifacts")
KENLM_DIR.mkdir(exist_ok=True)

train_corpus_path = KENLM_DIR / "train_corpus.txt"
bigram_arpa_path = KENLM_DIR / "bigram.arpa"
trigram_arpa_path = KENLM_DIR / "trigram.arpa"
bigram_binary_path = KENLM_DIR / "bigram.binary"
trigram_binary_path = KENLM_DIR / "trigram.binary"


def write_sentences_for_kenlm(sentences, output_path: Path):
    with output_path.open("w", encoding="utf-8") as f:
        for sentence in sentences:
            if sentence:
                f.write(" ".join(sentence) + "\n")


def build_kenlm_model(order: int, corpus_path: Path, arpa_path: Path, binary_path: Path):
    lmplz_command = [
        lmplz_path,
        "-o", str(order),
        "--text", str(corpus_path),
        "--arpa", str(arpa_path),
        "--discount_fallback",
    ]
    subprocess.run(lmplz_command, check=True)

    build_binary_command = [build_binary_path, str(arpa_path), str(binary_path)]
    subprocess.run(build_binary_command, check=True)


def kenlm_corpus_perplexity(model, sentences):
    total_log10_probability = 0.0
    total_predicted_tokens = 0

    for sentence in sentences:
        sentence_text = " ".join(sentence)
        total_log10_probability += model.score(sentence_text, bos=True, eos=True)
        total_predicted_tokens += len(sentence) + 1

    return 10 ** (-total_log10_probability / total_predicted_tokens)


write_sentences_for_kenlm(train_sentences, train_corpus_path)
build_kenlm_model(order=2, corpus_path=train_corpus_path, arpa_path=bigram_arpa_path, binary_path=bigram_binary_path)
build_kenlm_model(order=3, corpus_path=train_corpus_path, arpa_path=trigram_arpa_path, binary_path=trigram_binary_path)

kenlm_bigram_model = kenlm.Model(str(bigram_binary_path))
kenlm_trigram_model = kenlm.Model(str(trigram_binary_path))

kenlm_bigram_perplexity = kenlm_corpus_perplexity(kenlm_bigram_model, test_sentences)
kenlm_trigram_perplexity = kenlm_corpus_perplexity(kenlm_trigram_model, test_sentences)

if "perplexity_results" not in globals():
    perplexity_results = {
        model_name: model.perplexity(test_sentences)
        for model_name, model in language_models.items()
    }

our_bigram_perplexity = perplexity_results["bigram"]
our_trigram_perplexity = perplexity_results["trigram"]

print("KenLM perplexity on the testing dataset:")
print(f"kenlm bigram: {kenlm_bigram_perplexity:.4f}")
print(f"kenlm trigram: {kenlm_trigram_perplexity:.4f}")

print("\nComparison with my additive-smoothing implementation:")
print(f"my bigram:    {our_bigram_perplexity:.4f}")
print(f"kenlm bigram: {kenlm_bigram_perplexity:.4f}")
print(f"my trigram:   {our_trigram_perplexity:.4f}")
print(f"kenlm trigram:{kenlm_trigram_perplexity:.4f}")

print("\nComparison analysis:")
print("KenLM is a highly optimized toolkit, so its perplexity is often lower than a simple hand-written implementation.")
print("If KenLM performs better, it usually handles sparse statistics more effectively.")
print("If the gap is small, the hand-written model already captures the main local patterns reasonably well.")


## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [ ]:
from collections import defaultdict

if "processed_tokens" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_tokens" not in test_records[0]:
    attach_processed_text(test_records, nlp)


def normalize_document_tokens(tokens, vocabulary):
    return [token if token in vocabulary else UNK_TOKEN for token in tokens]


train_documents = [normalize_document_tokens(record["processed_tokens"], vocabulary) for record in train_records]
train_labels = [record["label"] for record in train_records]
test_documents = [normalize_document_tokens(record["processed_tokens"], vocabulary) for record in test_records]
test_labels = [record["label"] for record in test_records]


class MultinomialNaiveBayes:
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.classes_ = []
        self.vocabulary_ = set()
        self.class_document_counts = Counter()
        self.class_token_counts = Counter()
        self.token_counts_per_class = defaultdict(Counter)
        self.log_class_priors = {}
        self.log_token_likelihoods = defaultdict(dict)
        self.default_log_token_likelihood = {}

    def fit(self, documents, labels, vocabulary):
        self.vocabulary_ = set(vocabulary)
        self.classes_ = sorted(set(labels))
        vocabulary_size = len(self.vocabulary_)
        total_documents = len(documents)

        for document_tokens, label in zip(documents, labels):
            self.class_document_counts[label] += 1
            document_counter = Counter(document_tokens)
            for token, count in document_counter.items():
                self.token_counts_per_class[label][token] += count
                self.class_token_counts[label] += count

        for label in self.classes_:
            self.log_class_priors[label] = math.log(self.class_document_counts[label] / total_documents)

        for label in self.classes_:
            denominator = self.class_token_counts[label] + self.alpha * vocabulary_size
            self.default_log_token_likelihood[label] = math.log(self.alpha / denominator)
            for token in self.vocabulary_:
                probability = (self.token_counts_per_class[label][token] + self.alpha) / denominator
                self.log_token_likelihoods[label][token] = math.log(probability)

    def predict_one(self, document_tokens):
        document_counter = Counter(document_tokens)
        class_scores = {}

        for label in self.classes_:
            score = self.log_class_priors[label]
            for token, count in document_counter.items():
                token_log_probability = self.log_token_likelihoods[label].get(token, self.default_log_token_likelihood[label])
                score += count * token_log_probability
            class_scores[label] = score

        return max(class_scores, key=class_scores.get)

    def predict(self, documents):
        return [self.predict_one(document_tokens) for document_tokens in documents]


nb_classifier = MultinomialNaiveBayes(alpha=1.0)
nb_classifier.fit(train_documents, train_labels, vocabulary_list)
nb_predictions = nb_classifier.predict(test_documents)
nb_accuracy = sum(pred == gold for pred, gold in zip(nb_predictions, test_labels)) / len(test_labels)

print("Naive Bayes classifier has been trained and tested on the test dataset.")
print(f"Test accuracy: {nb_accuracy:.4f}")
print("\nFirst 10 prediction examples:")
for index, (predicted_label, gold_label, record) in enumerate(zip(nb_predictions, test_labels, test_records), start=1):
    if index > 10:
        break
    print(f"{index}. title={record['title']} | predicted={predicted_label} | gold={gold_label}")


> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

if "processed_text" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_text" not in test_records[0]:
    attach_processed_text(test_records, nlp)

train_texts = [record["processed_text"] for record in train_records]
train_labels = [record["label"] for record in train_records]
test_texts = [record["processed_text"] for record in test_records]
test_labels = [record["label"] for record in test_records]

lr_train_texts, lr_valid_texts, lr_train_labels, lr_valid_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels,
)

lr_param_grid = [
    {"ngram_range": (1, 1), "min_df": 3, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 3, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 5, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 3, "C": 2.0},
]

lr_validation_results = []
best_lr_score = -1.0
best_lr_config = None

for config in lr_param_grid:
    vectorizer = TfidfVectorizer(
        lowercase=False,
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
    )

    X_train = vectorizer.fit_transform(lr_train_texts)
    X_valid = vectorizer.transform(lr_valid_texts)

    lr_model = LogisticRegression(C=config["C"], max_iter=1000, solver="lbfgs", multi_class="auto")
    lr_model.fit(X_train, lr_train_labels)

    valid_predictions = lr_model.predict(X_valid)
    valid_accuracy = accuracy_score(lr_valid_labels, valid_predictions)
    valid_macro_f1 = f1_score(lr_valid_labels, valid_predictions, average="macro")

    lr_validation_results.append({
        "config": config,
        "valid_accuracy": valid_accuracy,
        "valid_macro_f1": valid_macro_f1,
    })

    if valid_macro_f1 > best_lr_score:
        best_lr_score = valid_macro_f1
        best_lr_config = config

final_lr_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=best_lr_config["ngram_range"],
    min_df=best_lr_config["min_df"],
)

X_train_full = final_lr_vectorizer.fit_transform(train_texts)
X_test = final_lr_vectorizer.transform(test_texts)

lr_classifier = LogisticRegression(C=best_lr_config["C"], max_iter=1000, solver="lbfgs", multi_class="auto")
lr_classifier.fit(X_train_full, train_labels)
lr_predictions = lr_classifier.predict(X_test)
lr_test_accuracy = accuracy_score(test_labels, lr_predictions)

print("Validation results for different LR settings:")
for result_row in lr_validation_results:
    print(
        f"config={result_row['config']} | "
        f"valid_accuracy={result_row['valid_accuracy']:.4f} | "
        f"valid_macro_f1={result_row['valid_macro_f1']:.4f}"
    )

print("\nBest LR configuration:")
print(best_lr_config)
print(f"Best validation macro-F1: {best_lr_score:.4f}")
print("\nFinal Logistic Regression test result:")
print(f"Test accuracy: {lr_test_accuracy:.4f}")
print("\nFirst 10 LR prediction examples:")
for index, (predicted_label, gold_label, record) in enumerate(zip(lr_predictions, test_labels, test_records), start=1):
    if index > 10:
        break
    print(f"{index}. title={record['title']} | predicted={predicted_label} | gold={gold_label}")


> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [ ]:
from sklearn.metrics import classification_report

if "nb_predictions" not in globals():
    raise RuntimeError("Please run the Naive Bayes cell first so that nb_predictions is available.")
if "lr_predictions" not in globals():
    raise RuntimeError("Please run the Logistic Regression cell first so that lr_predictions is available.")

nb_micro_f1 = f1_score(test_labels, nb_predictions, average="micro")
nb_macro_f1 = f1_score(test_labels, nb_predictions, average="macro")
lr_micro_f1 = f1_score(test_labels, lr_predictions, average="micro")
lr_macro_f1 = f1_score(test_labels, lr_predictions, average="macro")

print("F1-score results on the testing dataset:")
print(f"Naive Bayes    | Micro-F1: {nb_micro_f1:.4f} | Macro-F1: {nb_macro_f1:.4f}")
print(f"Logistic Reg.  | Micro-F1: {lr_micro_f1:.4f} | Macro-F1: {lr_macro_f1:.4f}")

print("\nNaive Bayes classification report:")
print(classification_report(test_labels, nb_predictions, digits=4))
print("Logistic Regression classification report:")
print(classification_report(test_labels, lr_predictions, digits=4))

print("Explanation:")
better_micro_model = "Naive Bayes" if nb_micro_f1 >= lr_micro_f1 else "Logistic Regression"
better_macro_model = "Naive Bayes" if nb_macro_f1 >= lr_macro_f1 else "Logistic Regression"
print(f"For Micro-F1, the better model is {better_micro_model}.")
print(f"For Macro-F1, the better model is {better_macro_model}.")
print("If Micro-F1 is much higher than Macro-F1, the model is usually better on large classes than on small classes.")
print("This is common here because the dataset is imbalanced.")


### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [1]:
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

try:
    from gensim.models import Word2Vec
except ImportError as exc:
    raise ImportError("gensim is required for Task4. Please install gensim before running this cell.") from exc

if "processed_tokens" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_tokens" not in test_records[0]:
    attach_processed_text(test_records, nlp)

task4_train_documents = [record["processed_tokens"] for record in train_records]
task4_train_labels = [record["label"] for record in train_records]
task4_test_documents = [record["processed_tokens"] for record in test_records]
task4_test_labels = [record["label"] for record in test_records]

TASK4_VECTOR_SIZE = 100
TASK4_WINDOW = 5
TASK4_MIN_COUNT = 2
TASK4_WORKERS = 1
TASK4_SEED = 42

task4_word2vec = Word2Vec(
    sentences=task4_train_documents,
    vector_size=TASK4_VECTOR_SIZE,
    window=TASK4_WINDOW,
    min_count=TASK4_MIN_COUNT,
    workers=TASK4_WORKERS,
    sg=1,
    seed=TASK4_SEED,
)


def document_to_average_embedding(document_tokens, word2vec_model):
    known_vectors = []
    for token in document_tokens:
        if token in word2vec_model.wv:
            known_vectors.append(word2vec_model.wv[token])
    if not known_vectors:
        return np.zeros(word2vec_model.vector_size, dtype=np.float32)
    return np.mean(known_vectors, axis=0)


def build_document_embedding_matrix(documents, word2vec_model):
    return np.vstack([
        document_to_average_embedding(document_tokens, word2vec_model)
        for document_tokens in documents
    ])


task4_train_embeddings = build_document_embedding_matrix(task4_train_documents, task4_word2vec)
task4_test_embeddings = build_document_embedding_matrix(task4_test_documents, task4_word2vec)

task4_X_train, task4_X_valid, task4_y_train, task4_y_valid = train_test_split(
    task4_train_embeddings,
    task4_train_labels,
    test_size=0.2,
    random_state=42,
    stratify=task4_train_labels,
)

task4_c_candidates = [0.5, 1.0, 2.0, 5.0]
task4_validation_results = []
task4_best_c = None
task4_best_macro_f1 = -1.0

for c_value in task4_c_candidates:
    candidate_model = LogisticRegression(C=c_value, max_iter=1000, solver="lbfgs", multi_class="auto")
    candidate_model.fit(task4_X_train, task4_y_train)

    valid_predictions = candidate_model.predict(task4_X_valid)
    valid_micro_f1 = f1_score(task4_y_valid, valid_predictions, average="micro")
    valid_macro_f1 = f1_score(task4_y_valid, valid_predictions, average="macro")

    task4_validation_results.append({
        "C": c_value,
        "valid_micro_f1": valid_micro_f1,
        "valid_macro_f1": valid_macro_f1,
    })

    if valid_macro_f1 > task4_best_macro_f1:
        task4_best_macro_f1 = valid_macro_f1
        task4_best_c = c_value

task4_classifier = LogisticRegression(C=task4_best_c, max_iter=1000, solver="lbfgs", multi_class="auto")
task4_classifier.fit(task4_train_embeddings, task4_train_labels)
task4_predictions = task4_classifier.predict(task4_test_embeddings)

task4_micro_f1 = f1_score(task4_test_labels, task4_predictions, average="micro")
task4_macro_f1 = f1_score(task4_test_labels, task4_predictions, average="macro")

print("Validation results for embedding-based Logistic Regression:")
for result_row in task4_validation_results:
    print(
        f"C={result_row['C']} | "
        f"valid_micro_f1={result_row['valid_micro_f1']:.4f} | "
        f"valid_macro_f1={result_row['valid_macro_f1']:.4f}"
    )

print("\nBest C selected from validation:")
print(task4_best_c)
print("\nEmbedding-based classification results on the testing dataset:")
print(f"Micro-F1: {task4_micro_f1:.4f}")
print(f"Macro-F1: {task4_macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(task4_test_labels, task4_predictions, digits=4))
print("Explanation:")
print("Documents are represented by the average of their word embeddings.")
print("This captures some semantic similarity, but still ignores word order.")
print("If Micro-F1 is higher than Macro-F1, the model is usually stronger on large classes than on small classes.")


#### upload your homework: use html file format
!jupyter nbconvert assignment-01-XXX-YYY.ipynb --to html --template classic --embed-images